## Identifications des éléments d'adresses 

In [ ]:
import pandas as pd
import re
import unidecode
from collections import Counter 


In [ ]:
df = pd.read_csv("../01_data/patients/202403_adresse_geocoded_brutes.csv", sep=";")

df.dropna(subset='adresse')

In [ ]:
df_simplified = df[['geocancer_id','requete','adresse','cpville','ville']]

In [ ]:
df_simplified['adresse'] = df_simplified['adresse'].astype(str).apply(lambda x: x.upper())
df_simplified['requete'] = df_simplified['requete'].astype(str).apply(lambda x: x.upper())

In [ ]:
voirie = ["RUE","COURS","COUR","VOIE","RUELLE","ESPLANADE",
          "PLACE","SQUARE","SQ","ROND POINT","PL",
          "IMPASSE", "ALLEE", "CHEMIN" ,"ROUTE","RTE","IMP","PROMENADE","ALLE","ALL",
          "AVENUE", "BOULEVARD","BD","AVE","BLD","BVD","BLV","AVN","BV",
          "FERME","DOMAINE","LIEU DIT","QUARTIER","QUR","AV"]

##ATT AVEC COUR : A LA FOIS DANS LES ADRESSES ET COMME VOIRIE   

numeros = ["1","2","3","4","5","6","7","8","9","0"]

bruit = [ "RESIDENCE","CHEZ","HOPITAL","SDF","RES","MME","BAT","MAISON","MR","HOTEL",
         "LOTISSEMENT","CENTRE","QUARTIER","APPT","APT","SANTE", "RETRAITE", "HOP","TRANSFERT"]

In [ ]:
def find_attribute(attributes, text, is_number=False):
    if pd.notna(text):
        if is_number:
            numbers = re.findall(r'\d+', text)
            
            return ','.join(numbers) if numbers else ""
        else:
            found_attributes = [x for x in attributes if re.search(r'\b' + re.escape(x) + r'\b', text)]

            return ','.join(found_attributes) if found_attributes else ""
        
        return ""
    

    
df_simplified['voirie'] = df_simplified.apply(lambda x: find_attribute(voirie, x['adresse']), axis=1)
df_simplified['numeros'] = df_simplified.apply(lambda x: find_attribute(numeros, x['adresse'],is_number=True), axis=1)
df_simplified['bruit'] = df_simplified.apply(lambda x: find_attribute(bruit, x['adresse']), axis=1)

In [ ]:

df_simplified['nom_voirie'] = ""

for i in df_simplified.index:

    num = df_simplified.loc[i,'numeros'].split()
    voirie = df_simplified.loc[i,'voirie'].split()
    adresse = df_simplified.loc[i,'adresse'].split(' ')

    adr = [x for x in adresse if x not in num and x not in voirie ]
    
    df_simplified.at[i,'nom_voirie'] = " ".join(adr)

df_simplified.head()

## On ne garde que les premiers numéros apparaissant dans l'adresse en supposant que cela corresponde au numéro de rue

In [ ]:
for i in df_simplified.index:
    nums = df_simplified.loc[i,"numeros"].split(',')
    if len(nums)>1:
        df_simplified.loc[i,"numeros"] = nums[0]
          

#df_simplified[df_simplified["numeros"].str.split(',').str.len()>1]


### Freq des éléménts dans nos données : 

In [ ]:
mots = df_simplified["requete"].str.split(expand=True).stack()
mot_sans_accents = [ unidecode.unidecode(mot) for mot in mots]

mots_up = [mot.upper() for mot in mot_sans_accents]

In [ ]:
count = Counter(mots_up) 
mots_freq = count.most_common()
co = pd.DataFrame(mots_freq, columns = ["token", "count"])

In [ ]:
res = co[co["count"]>500]

In [ ]:
import plotly.graph_objects as go
fig = go.Figure(data=[
                     go.Table(
                        header=dict(values=list(res.columns),align='center'),
                        cells=dict(values=res.values.transpose(),
                                   fill_color = [["white","lightgrey"]*res.shape[0]],
                                   align='center'
                                  )
                            )
                       ])
fig.update_layout(
    autosize=False,
    margin = {'l':0,'r':0,'t':0,'b':0},
    height = 600
)

#fig.show()
fig.write_image('./images/token_occurence.png', scale =1)

## % apparition des différentes classes de l'adresse

In [ ]:
street_w_num = len(df_simplified[df_simplified['numeros'] !=""])
prc_street_w_num = (street_w_num/len(df_simplified)) *100

street_w_type = len(df_simplified[df_simplified['voirie'] !=""])
prc_street_w_type = (street_w_num/len(df_simplified)) *100


street_w_name = len(df_simplified[df_simplified['nom_voirie'] !=""])
prc_street_w_name = (street_w_name/len(df_simplified)) *100


street_w_cpville = len(df_simplified[df_simplified['cpville'] !=""])
prc_street_w_cpville = (street_w_name/len(df_simplified)) *100

In [ ]:
prc_street_w_type

In [ ]:
df_simplified[df_simplified['voirie'] ==""]